In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
import pandas as pd

In [ ]:
baseline = pd.read_csv("/content/gdrive/MyDrive/Scores/Scores/baseline_FID.csv")
finetune = pd.read_csv("/content/gdrive/MyDrive/Scores/Scores/finetuned_FID.csv")

In [ ]:
merged = baseline.merge(finetune, on="Category", suffixes=("_baseline", "_finetune"))

In [ ]:
merged = merged.drop(columns=["SD_FID", "JUGGXL_FID", "DALLE2_FID"], errors="ignore")

In [ ]:
total_categories = len(merged)

print("DreamBooth beats Mitsua:",
      f"{(merged['DREAM_FID'] < merged['MITSUA_FID']).mean() * 100:.1f}%")

print("LoRA beats Mitsua:",
      f"{(merged['LORA_FID'] < merged['MITSUA_FID']).mean() * 100:.1f}%")

print("Textual Inversion beats Mitsua:",
      f"{(merged['TI_FID'] < merged['MITSUA_FID']).mean() * 100:.1f}%")

In [ ]:
dream_not_beat = merged.loc[
    merged["DREAM_FID"] >= merged["MITSUA_FID"],
    "Category"
]

lora_not_beat = merged.loc[
    merged["LORA_FID"] >= merged["MITSUA_FID"],
    "Category"
]

ti_not_beat = merged.loc[
    merged["TI_FID"] >= merged["MITSUA_FID"],
    "Category"
]

In [ ]:
# Categories where each method did NOT beat Mitsua
dream_not_beat = set(merged.loc[merged["DREAM_FID"] >= merged["MITSUA_FID"], "Category"])
lora_not_beat = set(merged.loc[merged["LORA_FID"] >= merged["MITSUA_FID"], "Category"])
ti_not_beat = set(merged.loc[merged["TI_FID"] >= merged["MITSUA_FID"], "Category"])

all_categories = set(merged["Category"])

# Overlap across all three
overlap_all = dream_not_beat & lora_not_beat & ti_not_beat

print("Categories where all three underperformed:")
print(sorted(overlap_all))
print(f"Count: {len(overlap_all)}")
print(f"Percentage of all categories: {len(overlap_all) / len(all_categories) * 100:.1f}%")

# **Why they underperform?**

In [ ]:
import os
import glob
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from transformers import CLIPProcessor, CLIPModel
from scipy.stats import spearmanr, pearsonr

In [ ]:
eval_root = "/content/gdrive/MyDrive/DATA/COCO/Validation/Images"
finetune_root = "/content/gdrive/MyDrive/DATA/COCO/Fine-Tuning/Images"

image_exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

clip_model.eval()

In [ ]:
def get_image_paths(folder):
    paths = []
    for ext in image_exts:
        paths.extend(glob.glob(os.path.join(folder, f"*{ext}")))
        paths.extend(glob.glob(os.path.join(folder, f"*{ext.upper()}")))
    return sorted(paths)


@torch.no_grad()
def embed_images(image_paths, batch_size=32):
    embeddings = []

    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[i:i + batch_size]
        images = []

        for path in batch_paths:
            try:
                img = Image.open(path).convert("RGB")
                images.append(img)
            except Exception as e:
                print(f"Skipping {path}: {e}")

        if len(images) == 0:
            continue

        inputs = clip_processor(
            images=images,
            return_tensors="pt",
            padding=True
        ).to(device)

        outputs = clip_model.get_image_features(**inputs)

        # Handle both CLIPModel and CLIPVisionModel output formats
        if hasattr(outputs, "pooler_output"):
            image_features = outputs.pooler_output
        else:
            image_features = outputs

        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        embeddings.append(image_features.cpu().numpy())

    return np.vstack(embeddings)

def mean_max_similarity(finetune_embs, eval_embs):
    sim_matrix = cosine_similarity(eval_embs, finetune_embs)
    max_sim_per_eval = sim_matrix.max(axis=1)
    return max_sim_per_eval.mean()


def mean_pairwise_similarity(finetune_embs, eval_embs):
    sim_matrix = cosine_similarity(eval_embs, finetune_embs)
    return sim_matrix.mean()


def centroid_similarity(finetune_embs, eval_embs):
    return cosine_similarity(
        eval_embs.mean(axis=0, keepdims=True),
        finetune_embs.mean(axis=0, keepdims=True)
    )[0, 0]

In [ ]:
def normalize_category(name):
    return name.replace("_", " ").strip().lower()

categories = sorted([
    d for d in os.listdir(eval_root)
    if os.path.isdir(os.path.join(eval_root, d))
])

# The fine-tuning dataset's category folders use underscores between words
# (e.g. "hot_dog") while the validation set's use spaces (e.g. "hot dog"),
# so match them by normalized name rather than assuming identical folder names.
finetune_folders = {
    normalize_category(d): d
    for d in os.listdir(finetune_root)
    if os.path.isdir(os.path.join(finetune_root, d))
}

similarity_rows = []

for category in categories:
    normalized = normalize_category(category)

    if normalized not in finetune_folders:
        print(f"Skipping {category}: no matching fine-tuning folder")
        continue

    eval_folder = os.path.join(eval_root, category)
    finetune_folder = os.path.join(finetune_root, finetune_folders[normalized])

    eval_paths = get_image_paths(eval_folder)
    finetune_paths = get_image_paths(finetune_folder)

    print(f"\nCategory: {category}")
    print(f"Eval images: {len(eval_paths)}")
    print(f"Fine-tuning images: {len(finetune_paths)}")

    if len(eval_paths) == 0 or len(finetune_paths) == 0:
        print(f"Skipping {category}: missing images")
        continue

    eval_embs = embed_images(eval_paths)
    finetune_embs = embed_images(finetune_paths)

    similarity_rows.append({
        "Category": category,
        "num_eval_images": len(eval_paths),
        "num_finetune_images": len(finetune_paths),
        "mean_max_clip_similarity": mean_max_similarity(finetune_embs, eval_embs),
        "mean_pairwise_clip_similarity": mean_pairwise_similarity(finetune_embs, eval_embs),
        "centroid_clip_similarity": centroid_similarity(finetune_embs, eval_embs),
    })

similarity_df = pd.DataFrame(similarity_rows)
similarity_df

In [ ]:
similarity_df.to_csv("/content/gdrive/MyDrive/Scores/Scores/category_train_eval_clip_similarity.csv", index=False)

# **Analyze Similarity**

In [ ]:
analysis_df = merged.merge(similarity_df, on="Category", how="inner")
analysis_df.head()

In [ ]:
for method in ["DREAM", "LORA", "TI"]:
    analysis_df[f"{method}_FID_improvement"] = (
        analysis_df["MITSUA_FID"] - analysis_df[f"{method}_FID"]
    )

    analysis_df[f"{method}_FID_improvement_pct"] = (
        (analysis_df["MITSUA_FID"] - analysis_df[f"{method}_FID"])
        / analysis_df["MITSUA_FID"]
        * 100
    )

In [ ]:
from statsmodels.stats.multitest import multipletests

similarity_col = "mean_max_clip_similarity"
methods = ["DREAM", "LORA", "TI"]

results = []
for method in methods:
    x = analysis_df[similarity_col]
    y = analysis_df[f"{method}_FID_improvement_pct"]

    spearman_corr, spearman_p = spearmanr(x, y)
    pearson_corr, pearson_p = pearsonr(x, y)

    results.append({
        "method": method,
        "spearman_corr": spearman_corr,
        "spearman_p": spearman_p,
        "pearson_corr": pearson_corr,
        "pearson_p": pearson_p,
    })

# Holm-Bonferroni correction across the three Spearman tests (one per fine-tuning
# method), since they're tested against the same hypothesis (does representativeness
# predict FID improvement) and reported together.
raw_pvals = [r["spearman_p"] for r in results]
reject, corrected_pvals, _, _ = multipletests(raw_pvals, method="holm", alpha=0.05)

for r, corrected_p, significant in zip(results, corrected_pvals, reject):
    r["spearman_p_holm"] = corrected_p
    r["spearman_significant_holm"] = significant

    print(f"\n{r['method']}")
    print(f"Spearman correlation: {r['spearman_corr']:.3f}, p={r['spearman_p']:.4f}, "
          f"holm-corrected p={r['spearman_p_holm']:.4f} "
          f"({'significant' if r['spearman_significant_holm'] else 'not significant'} at alpha=0.05)")
    print(f"Pearson correlation: {r['pearson_corr']:.3f}, p={r['pearson_p']:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression

similarity_col = "mean_max_clip_similarity"

for method in ["DREAM", "LORA", "TI"]:
    x = analysis_df[similarity_col].values.reshape(-1, 1)
    y = analysis_df[f"{method}_FID_improvement_pct"].values

    rho, p = spearmanr(
        analysis_df[similarity_col],
        analysis_df[f"{method}_FID_improvement_pct"]
    )

    plt.figure(figsize=(6.5, 4.5))

    # Scatter
    plt.scatter(
        analysis_df[similarity_col],
        analysis_df[f"{method}_FID_improvement_pct"],
        alpha=0.75,
        s=45
    )

    # Regression line
    model = LinearRegression()
    model.fit(x, y)

    x_line = np.linspace(x.min(), x.max(), 100).reshape(-1, 1)
    y_line = model.predict(x_line)

    plt.plot(
        x_line,
        y_line,
        linewidth=2
    )

    # Zero improvement line
    plt.axhline(0, linestyle="--", linewidth=1, alpha=0.6)

    plt.xlabel("Train-eval CLIP similarity")
    plt.ylabel("FID improvement over Mitsua (%)")
    plt.title(f"{method}: ρ = {rho:.3f}, p = {p:.4f}")

    plt.tight_layout()
    plt.show()